# RWY world life list × AviList

This notebook has two parts: **Part 1** crosswalks a personal eBird world life list against **AviList v2025**
(completion, taxonomy, evolution, conservation). **Part 2** explores **total birding history** from a My eBird export
(checklists, geography, overlaps, PCA). The companion `avilist_birds_explore.ipynb` covers AviList-wide themes.


In [ ]:
from __future__ import annotations

import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

try:
    from IPython.display import HTML, display
except ImportError:
    display = print
    HTML = lambda x: x


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if not (cand / "requirements.txt").exists():
            continue
        if (cand / "python" / "birds_nb.py").is_file():
            return cand
        if (cand / "birds_nb.py").is_file():
            return cand
    return p


REPO_ROOT = _repo_root()
_py = REPO_ROOT / "python"
if (_py / "birds_nb.py").is_file():
    sys.path.insert(0, str(_py))

from birds_nb import (
    CONTINENT_DISPLAY,
    collapse_sunburst_genera_by_family,
    family_label,
    genus_label,
    order_label,
    sp_region_label,
    sunburst_panzoom_viewport,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 130

DATA_DIR = REPO_ROOT / "data" if (REPO_ROOT / "data").is_dir() else REPO_ROOT
AVILIST_XLSX = DATA_DIR / "AviList-v2025-11Jun-extended.xlsx"
LIFELIST_CSV = DATA_DIR / "RWY_ebird_world_life_list.csv"
CACHE = DATA_DIR / ".cache_avilist.pkl.gz"
for p in (AVILIST_XLSX, LIFELIST_CSV):
    assert p.exists(), f"Missing {p}"
print(f"AviList {AVILIST_XLSX.name} ({AVILIST_XLSX.stat().st_size/1e6:.1f} MB) | list {LIFELIST_CSV.name} ({LIFELIST_CSV.stat().st_size/1e3:.1f} KB)")
# Geography choropleth: set EBIRD_API_KEY (free) — https://ebird.org/api/keygen


### Data loading and preparation

In [ ]:
from birds_nb import (
    add_genus_common_example,
    coalesce_species_english,
    continents_in,
    countries_in,
    load_avilist,
    regions_in,
)

YEAR_RE = re.compile(r"(1[5-9]\d{2}|20\d{2})")

def _yr(v):
    if not isinstance(v, str):
        return np.nan
    m = YEAR_RE.search(v)
    return float(m[1]) if m else np.nan

# Load + derive all Step 1 working columns in one place.
df_all = load_avilist(AVILIST_XLSX, CACHE)
df_all["Description_year"] = df_all["Authority"].map(_yr)
df_all["Genus"] = df_all["Scientific_name"].astype(str).str.split().str[0]

iucn_order = ["LC", "NT", "VU", "EN", "CR", "EW", "EX", "DD", "NE"]
raw_iucn = df_all["IUCN_Red_List_Category"].fillna("NE").astype(str).str.strip()
raw_iucn = raw_iucn.str.replace(r"^CR.*", "CR", regex=True).where(raw_iucn.isin(iucn_order), "NE")
df_all["IUCN"] = pd.Categorical(raw_iucn, categories=iucn_order, ordered=True)

extinct_raw = df_all["Extinct_or_possibly_extinct"].astype(str).str.strip().str.lower()
df_all["is_extinct"] = extinct_raw.isin({"extinct", "possibly extinct", "yes", "true", "1"}) | df_all["IUCN"].isin(["EX", "EW"])

df_species = df_all[df_all["Taxon_rank"] == "species"].copy().reset_index(drop=True)
df_species = add_genus_common_example(df_species)
df_family = df_all[df_all["Taxon_rank"] == "family"].copy().reset_index(drop=True)
df_order = df_all[df_all["Taxon_rank"] == "order"].copy().reset_index(drop=True)

df_species["Range_continents"] = df_species["Range"].map(continents_in)
df_species["Range_regions"] = df_species["Range"].map(regions_in)
df_species["Range_countries"] = df_species["Range"].map(countries_in)
df_species["N_continents"] = df_species["Range_continents"].map(len)

parsed = (df_species["N_continents"] > 0).sum()
country_parsed = (df_species["Range_countries"].map(len) > 0).sum()
print(
    f"Loaded {len(df_all):,} rows | orders {len(df_order)} | families {len(df_family)} | "
    f"genera {df_species['Genus'].nunique():,} | species {len(df_species):,} | "
    f"continent parsed {parsed:,}/{len(df_species):,} | "
    f"country parsed {country_parsed:,}/{len(df_species):,}"
)


## Part 1 — RWY life list × AviList

Now that I've imported all of the species metadata from the extended AviList, I wanted to see what I could learn about my personal life list which, at the time of writing this, sits at 1274 species.


In [ ]:
life = pd.read_csv(LIFELIST_CSV)
life.columns = [c.strip() for c in life.columns]
life["Date"] = pd.to_datetime(life["Date"], format="%d %b %Y", errors="coerce")
life = life.rename(columns={"Scientific Name": "Scientific_name", "Common Name": "Common_name"})
cols = ["Scientific_name", "Common_name", "Location", "S/P", "Date"]
matched = df_species.merge(life[cols], on="Scientific_name", how="left", indicator=True)
m = matched["_merge"] == "both"
first_seen = matched[m].sort_values("Date").groupby("Scientific_name", as_index=False).first()[cols]
df_species_life = df_species.merge(first_seen, on="Scientific_name", how="left")
df_species_life["seen"] = df_species_life["Date"].notna()
n_seen = int(df_species_life["seen"].sum())
miss = life[~life["Scientific_name"].isin(df_species["Scientific_name"])]


As a first pass, I generated another interactive sunburst plot and colored it this time by the percentage of species seen in each family.

In [ ]:
fam_pct = (
    df_species_life.groupby(["Order", "Family", "Family_English_name"], dropna=False)
    .agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
)
fam_pct["pct_seen"] = fam_pct["n_seen"] / fam_pct["n_species"] * 100
fam_pct["Order_plot"] = fam_pct["Order"].map(order_label)
fam_pct["Family_plot"] = [family_label(f, e) for f, e in zip(fam_pct["Family"], fam_pct["Family_English_name"])]

life_sb = (
    df_species_life.groupby(["Order", "Family", "Family_English_name", "Genus", "Genus_common_example"], dropna=False)
    .agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
)
life_sb["pct_seen"] = (100.0 * life_sb["n_seen"] / life_sb["n_species"]).replace([np.inf, -np.inf], np.nan).fillna(0.0)
life_sb = collapse_sunburst_genera_by_family(life_sb, max_genera_per_family=40, extra_sum_cols=("n_seen",))
life_sb["Order_plot"] = life_sb["Order"].map(order_label)
life_sb["Family_plot"] = [family_label(f, e) for f, e in zip(life_sb["Family"], life_sb["Family_English_name"])]
life_sb["Genus_plot"] = [genus_label(g, h) for g, h in zip(life_sb["Genus"], life_sb["Genus_common_example"])]
_path = ["Order_plot", "Family_plot", "Genus_plot"]
life_sb = (
    life_sb.groupby(_path, as_index=False)
    .agg(
        n_species=("n_species", "sum"),
        n_seen=("n_seen", "sum"),
        Order=("Order", "first"),
        Family=("Family", "first"),
        Family_English_name=("Family_English_name", "first"),
        Genus=("Genus", "first"),
        Genus_common_example=("Genus_common_example", "first"),
    )
)
life_sb["pct_seen"] = (100.0 * life_sb["n_seen"] / life_sb["n_species"].replace(0, np.nan)).fillna(0.0)

fig = px.sunburst(
    life_sb,
    path=_path,
    values="n_species",
    color="pct_seen",
    color_continuous_scale="YlOrRd",
    range_color=(0, 100),
    title="Life list % seen (Order→Family→Genus)",
    width=900,
    height=900,
)

trace = fig.data[0]
id_to_parent = dict(zip(trace.ids, trace.parents))
id_to_label = dict(zip(trace.ids, trace.labels))

orders, families, genera = [], [], []
for node_id in trace.ids:
    parent_id = id_to_parent.get(node_id, "")
    if not parent_id:
        order_plot = id_to_label.get(node_id, "")
        family_plot = ""
        genus_plot = ""
    else:
        grandparent_id = id_to_parent.get(parent_id, "")
        if not grandparent_id:
            order_plot = id_to_label.get(parent_id, "")
            family_plot = id_to_label.get(node_id, "")
            genus_plot = ""
        else:
            order_plot = id_to_label.get(grandparent_id, "")
            family_plot = id_to_label.get(parent_id, "")
            genus_plot = id_to_label.get(node_id, "")

    orders.append(order_plot)
    families.append(family_plot)
    genera.append(genus_plot)


def _life_sb_seen_total(order_plot, family_plot, genus_plot):
    d = life_sb
    if genus_plot:
        m = (d["Order_plot"] == order_plot) & (d["Family_plot"] == family_plot) & (d["Genus_plot"] == genus_plot)
        sub = d.loc[m]
    elif family_plot:
        sub = d.loc[(d["Order_plot"] == order_plot) & (d["Family_plot"] == family_plot)]
    elif order_plot:
        sub = d.loc[d["Order_plot"] == order_plot]
    else:
        sub = d
    if sub.empty:
        return 0, 0
    return int(sub["n_seen"].sum()), int(sub["n_species"].sum())


seen_lines = []
for o, f, g in zip(orders, families, genera):
    nv, ns = _life_sb_seen_total(o, f, g)
    pct = 100.0 * nv / ns if ns else 0.0
    seen_lines.append(f"% seen: {pct:.0f}% ({nv}/{ns})")

fig.update_traces(
    customdata=list(zip(orders, families, genera, seen_lines)),
    hovertemplate=(
        "Order: %{customdata[0]}<br>"
        "Family: %{customdata[1]}<br>"
        "Genus: %{customdata[2]}<br>"
        "%{customdata[3]}"
        "<extra></extra>"
    ),
)

fig.update_layout(
    dragmode="pan",
    margin=dict(t=65, l=30, r=30, b=30),
    uirevision="sunburst-life-pct",
    transition=dict(duration=0),
)
_cfg = {"scrollZoom": True, "displayModeBar": True, "doubleClick": "reset", "responsive": False}
SUNBURST_GD_ID = "sunburst-life-avilist"
fig_html = pio.to_html(fig, include_plotlyjs="cdn", full_html=False, config=_cfg, div_id=SUNBURST_GD_ID)
display(HTML(sunburst_panzoom_viewport(fig_html, SUNBURST_GD_ID, 900, 900)))


### Family tree (OpenTree) — order ring + life-list completion fill

Same interactive viewer as `avilist_birds_explore.ipynb`: pan/zoom, hover tooltips, **click a family** for a species-level cladogram. Each family tip uses a **coloured ring** for its bird order and a **grayscale inner disc**: white = 0% of that family’s AviList species on your life list, black = 100%, with shades in between (drill-down: **seen = solid**, not seen = faint).


In [ ]:
from phylo import (
    build_family_tree,
    build_family_subtrees,
    load_family_subtrees_inline,
    display_phylocanvas,
)
from birds_life_phylo import tint_family_meta_by_completion, tint_subtrees_by_seen

PHYLO_DIR = DATA_DIR / "phylogeny"
fam_nwk, fam_meta = build_family_tree(df_species, PHYLO_DIR)
build_family_subtrees(df_species, PHYLO_DIR)
_subtrees = load_family_subtrees_inline(PHYLO_DIR)

_pct = fam_pct.set_index("Family")["pct_seen"].to_dict()
_ns = fam_pct.set_index("Family")["n_seen"].astype(int).to_dict()
_nt = fam_pct.set_index("Family")["n_species"].astype(int).to_dict()
_seen_sci = set(df_species_life.loc[df_species_life["seen"], "Scientific_name"].astype(str))

_fam_meta_shaded = tint_family_meta_by_completion(
    fam_meta, _pct, _ns, _nt, df_species=df_species
)
_subtrees_shaded = tint_subtrees_by_seen(_subtrees, _seen_sci)
display_phylocanvas(
    fam_nwk,
    _fam_meta_shaded,
    "phylo-life-tree",
    height=760,
    drilldown=True,
    subtrees_inline=_subtrees_shaded,
)



### Most threatened species on the life list — where first seen

IUCN **VU / EN / CR / EW / EX** (mapped from AviList). Markers use coordinates from **My eBird** export when available.


In [ ]:
import plotly.graph_objects as go

_iucn_colors = {"LC": "#60c060", "NT": "#cfd862", "VU": "#f2c64e", "EN": "#ef8a3b", "CR": "#d83333", "EW": "#6f21a0", "EX": "#2a2a2a", "DD": "#a0a0a0", "NE": "#dedede"}
_threat_rank = {"VU": 1, "EN": 2, "CR": 3, "EW": 4, "EX": 5}

_mraw = DATA_DIR / "personal_ebird" / "raw" / "MyEBirdData.csv"
_coords = None
if _mraw.is_file():
    _me = pd.read_csv(_mraw)
    _me.columns = [str(c).strip() for c in _me.columns]
    if "Scientific Name" in _me.columns and "Latitude" in _me.columns and "Longitude" in _me.columns:
        _me["Latitude"] = pd.to_numeric(_me["Latitude"], errors="coerce")
        _me["Longitude"] = pd.to_numeric(_me["Longitude"], errors="coerce")
        if "Date" in _me.columns:
            _me["Date"] = pd.to_datetime(_me["Date"], errors="coerce")
        _me = _me.dropna(subset=["Latitude", "Longitude"])
        if "Date" in _me.columns:
            _me = _me.sort_values("Date", na_position="last")
        _first = _me.groupby("Scientific Name", as_index=False).first()
        _loc_col = next(
            (c for c in ("Location", "Location name", "Location Name", "Locality", "LOCALITY") if c in _me.columns),
            None,
        )
        _coords = _first[["Scientific Name", "Latitude", "Longitude"]].copy()
        if _loc_col is not None:
            _coords["eBird_locality"] = _first[_loc_col].astype(str)
        else:
            _coords["eBird_locality"] = ""

_end = df_species_life[
    df_species_life["seen"]
    & df_species_life["IUCN"].astype(str).isin(["VU", "EN", "CR", "EW", "EX"])
].copy()
_eng_df = df_species[["Scientific_name"]].copy()
_eng_df["English_common_merged"] = coalesce_species_english(df_species)
_end = _end.merge(_eng_df, on="Scientific_name", how="left")
if _coords is not None:
    _end = _end.merge(
        _coords.rename(columns={"Scientific Name": "Scientific_name"}),
        on="Scientific_name",
        how="left",
    )
else:
    _end["Latitude"] = np.nan
    _end["Longitude"] = np.nan
    _end["eBird_locality"] = ""

if "Location" not in _end.columns:
    _end["Location"] = ""
if "S/P" not in _end.columns:
    _end["S/P"] = ""
# Life list + My eBird both may carry locality names; prefer first-seen row from eBird export.
_end["_hover_locality"] = _end["eBird_locality"].fillna(_end["Location"]).fillna("").astype(str)

_cn = _end["Common_name"].fillna(_end["English_common_merged"]).fillna("")
_end["lab"] = (_cn.astype(str) + " (" + _end["Scientific_name"].astype(str) + ")").str.strip()
_end["iucn_s"] = _end["IUCN"].astype(str)
_end["msize"] = _end["iucn_s"].map(lambda x: 8 + 5 * _threat_rank.get(x, 0))

_geo = go.Figure()
_sub = _end.dropna(subset=["Latitude", "Longitude"])
if not _sub.empty:
    for cat, g in _sub.groupby("iucn_s"):
        _geo.add_trace(
            go.Scattergeo(
                lat=g["Latitude"],
                lon=g["Longitude"],
                mode="markers",
                name=cat,
                marker=dict(
                    size=g["msize"],
                    color=_iucn_colors.get(cat, "#888888"),
                    line=dict(width=0.4, color="#222"),
                ),
                text=g["lab"],
                customdata=np.column_stack(
                    [
                        g["Scientific_name"].astype(str),
                        g["Date"].fillna(pd.NaT).astype(str),
                        g["_hover_locality"].astype(str),
                        g["S/P"].fillna("").astype(str),
                        np.repeat(str(cat), len(g)),
                    ]
                ),
                hovertemplate=(
                    "<b>%{text}</b><br>IUCN: %{customdata[4]}<br>First seen: %{customdata[1]}<br>"
                    "Location: %{customdata[2]}<br>S/P: %{customdata[3]}<extra></extra>"
                ),
            )
        )
else:
    _geo.add_annotation(
        text="No coordinates — add My eBird export under data/personal_ebird/raw/",
        showarrow=False,
    )

_geo.update_layout(
    title="Threatened / extinct-in-wild species on the life list (first-seen coordinates)",
    height=520,
    margin=dict(l=10, r=10, t=55, b=10),
    geo=dict(projection_type="natural earth", showland=True, landcolor="#f0f0f0"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
_geo.show()

_order_cat = pd.Categorical(_end["IUCN"], categories=["CR", "EN", "VU", "EW", "EX"], ordered=True)
_end["_sort"] = _order_cat
_bar = _end.sort_values(["_sort", "Scientific_name"], ascending=[True, True]).head(40)
_y = []
_cols = []
for _, r in _bar.iterrows():
    cn = r.get("Common_name") or r.get("English_common_merged") or ""
    sp = r.get("S/P", "?")
    _y.append(f"{cn} ({r['Scientific_name']}) — {sp}")
    _cols.append(_iucn_colors.get(str(r["IUCN"]), "#888"))
if _y:
    fig_b, ax_b = plt.subplots(figsize=(11, max(4, len(_y) * 0.22)))
    ax_b.barh(
        _y,
        width=1.0,
        color=_cols,
    )
    ax_b.set_title("Threatened species on the life list (top 40)")
    ax_b.set_xlim(0, 1.15)
    ax_b.set_xticks([])
    plt.tight_layout()
    plt.show()
else:
    print("No VU+ species on the merged life list.")



In [ ]:
ord_stats = df_species_life.groupby("Order").agg(n_species=("seen", "size"), n_seen=("seen", "sum")).reset_index()
ord_stats["pct_seen"] = ord_stats["n_seen"] / ord_stats["n_species"] * 100
ord_plot = ord_stats.sort_values("pct_seen", ascending=True)
yl = ord_plot["Order"].map(order_label)
fig, ax = plt.subplots(figsize=(11, 12))
ax.barh(yl, ord_plot["pct_seen"], color=sns.color_palette("crest", n_colors=len(ord_plot)))
for i, (p, n, s) in enumerate(zip(ord_plot["pct_seen"], ord_plot["n_species"], ord_plot["n_seen"])):
    ax.text(p + 0.5, i, f"{s}/{n}", va="center", fontsize=7)
ax.set(xlabel="% order seen", xlim=(0, 100), title="Completion by order")
plt.tight_layout()
plt.show()


In [ ]:
untouched = ord_stats.query("n_seen == 0").sort_values("n_species", ascending=False).copy()
untouched.insert(0, "Order (readable)", untouched["Order"].map(order_label))
print(f"No species yet in {len(untouched)} orders:")
untouched


In [ ]:
min_size = 10
completion = fam_pct.query("n_species >= @min_size").sort_values("pct_seen", ascending=False).head(25).iloc[::-1]
labels = [f"{family_label(r.Family, r.Family_English_name)}  ({int(r.n_seen)}/{int(r.n_species)})" for r in completion.itertuples(index=False)]
fig, ax = plt.subplots(figsize=(11, 9))
bars = ax.barh(labels, completion["pct_seen"], color=sns.color_palette("crest", n_colors=len(completion)))
ax.set(xlabel="% family seen", xlim=(0, 100), title=f"Most complete families (≥{min_size} sp.)")
for b, pct in zip(bars, completion["pct_seen"]):
    ax.text(pct + 0.5, b.get_y() + b.get_height() / 2, f"{pct:.0f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
accum = df_species_life.dropna(subset=["Date"]).sort_values("Date").assign(cum=lambda d: range(1, len(d) + 1))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(accum["Date"], accum["cum"], color="#3d7ea0", lw=1.5)
ax.fill_between(accum["Date"], accum["cum"], alpha=0.15, color="#3d7ea0")
ax.set(xlabel="date", ylabel="cumulative species", title=f"Species accumulation ({int(accum['cum'].iloc[-1])})")
plt.tight_layout()
plt.show()


In [ ]:
by_region = df_species_life.dropna(subset=["Date"]).groupby("S/P").agg(n=("Scientific_name", "size"), n_families=("Family", "nunique")).sort_values("n", ascending=False).reset_index()
print(f"{len(by_region)} eBird regions")
top = by_region.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(top["S/P"].map(sp_region_label), top["n"], color="#3d7ea0")
for i, (n, f) in enumerate(zip(top["n"], top["n_families"])):
    ax.text(n + 1, i, f"{n} sp / {f} fam", va="center", fontsize=8)
ax.set(xlabel="species (first seen in region)", title="Top 15 regions (eBird S/P codes)")
plt.tight_layout()
plt.show()


In [ ]:
rf = df_species_life.dropna(subset=["Date"]).groupby(["S/P", "Family"]).size().unstack(fill_value=0)
top_regions = by_region["S/P"].head(15).tolist()
top_fams = df_species_life.dropna(subset=["Date"])["Family"].value_counts().head(25).index.tolist()
rf_sub = rf.reindex(index=top_regions, columns=top_fams).fillna(0)
fe = df_family.set_index("Scientific_name")["Family_English_name"].to_dict()
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(rf_sub, cmap="YlGnBu", annot=True, fmt=".0f", cbar_kws={"label": "species"}, lw=0.3, ax=ax, xticklabels=[family_label(f, fe.get(f, "")) for f in rf_sub.columns], yticklabels=[sp_region_label(r) for r in rf_sub.index])
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
ax.set_title("Region × family lifers")
plt.tight_layout()
plt.show()


In [ ]:
dom = (
    df_species_life.dropna(subset=["Date"]).groupby(["S/P", "Family"]).size().reset_index(name="n")
    .sort_values(["S/P", "n"], ascending=[True, False]).drop_duplicates("S/P").sort_values("n", ascending=False).head(25).reset_index(drop=True)
)
dom


## Part 2 — Total birding history

Save **My eBird → Download my data** exports under `data/personal_ebird/raw/`. Helpers in `ebird_personal` write
`data/personal_ebird/by_country/{CC}/observations.csv`. If `raw/` is empty, the next cell falls back to the world life list CSV as a small demo.

The cells below load the export into `hist`, print high-level EDA, map your species by country (with the same family search UX as `avilist_birds_explore.ipynb`), compare regional country overlaps, and run a PCA of countries by species composition.

With `EBIRD_API_KEY`, the loader cell also caches sample checklist JSON and reference `spplist` counts for a few countries.


In [ ]:
import os

from ebird_personal import (
    add_country_column,
    checklist_observations_table,
    checklist_summary_rows,
    fetch_checklist_view_json,
    load_my_ebird_export,
    partition_by_country,
    reference_spplists_for_iso2,
)
from ebird_spatial import load_ebird_taxonomy

PERSONAL_DIR = DATA_DIR / "personal_ebird"
RAW_DIR = PERSONAL_DIR / "raw"
BY_COUNTRY_DIR = PERSONAL_DIR / "by_country"
EBIRD_CACHE = DATA_DIR / ".cache_ebird"
RAW_DIR.mkdir(parents=True, exist_ok=True)
BY_COUNTRY_DIR.mkdir(parents=True, exist_ok=True)

_exports = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.txt")) + sorted(RAW_DIR.glob("*.tsv"))
_export_path = _exports[0] if _exports else LIFELIST_CSV
_export_df = load_my_ebird_export(_export_path)
_country_files = partition_by_country(_export_df, BY_COUNTRY_DIR)
_cl_summary = checklist_summary_rows(_export_df)
try:
    _rel = _export_path.relative_to(REPO_ROOT)
except ValueError:
    _rel = _export_path
print(
    f"Source {_rel} | rows={len(_export_df):,} | countries={len(_country_files)} | "
    f"unique checklists={len(_cl_summary):,}"
)

# Full export as a working table for Part 2 visualizations
hist = add_country_column(_export_df.copy())
if "Date" in hist.columns:
    # My eBird bulk CSV uses ISO dates (2025-10-23); life-list-style rows often use "05 Apr 2026".
    hist["Date"] = pd.to_datetime(hist["Date"], errors="coerce")
elif "OBSERVATION DATE" in hist.columns:
    hist["Date"] = pd.to_datetime(hist["OBSERVATION DATE"], errors="coerce")
print(f"hist rows={len(hist):,} | columns={len(hist.columns)}")

_key = os.environ.get("EBIRD_API_KEY", "").strip()
if _key:
    _tax = load_ebird_taxonomy(EBIRD_CACHE)
    _sid_col = next(
        (c for c in _cl_summary.columns if c in ("SubID", "SUBMISSION ID", "Submission ID", "subId")),
        None,
    )
    if _sid_col is not None and not _cl_summary.empty:
        _sample_sid = str(_cl_summary.iloc[0][_sid_col]).strip()
        _j = fetch_checklist_view_json(_sample_sid, _key, EBIRD_CACHE)
        _obs_tbl = checklist_observations_table(_j, _tax)
        print(f"Sample checklist {_sample_sid}: {len(_obs_tbl)} taxon rows (with sciName)")
    _ccodes = sorted(
        {str(x).upper() for x in _cl_summary["country_iso2"].dropna().tolist() if len(str(x)) == 2}
    )[:5]
    _ref = reference_spplists_for_iso2(_ccodes, EBIRD_CACHE, _key)
    print("Reference spplist species-code counts (first 5 countries):", {k: len(v) for k, v in _ref.items()})
else:
    print("Optional: set EBIRD_API_KEY for checklist JSON + reference spplists (see README).")


### EDA — checklists, species, and locations


In [ ]:
from birds_history import summarise_history

_k = summarise_history(hist)
print(
    "Rows:",
    f"{_k['n_rows']:,}",
    "| checklists:",
    f"{_k['n_checklists']:,}",
    "| locations:",
    f"{_k['n_locations']:,}",
    "| countries:",
    _k["n_countries"],
    "| distinct species:",
    _k["n_species"],
)
if _k.get("date_min") is not None:
    print("Date span:", _k["date_min"], "→", _k["date_max"])

_sid = next(
    (c for c in ("SubID", "SUBMISSION ID", "Submission ID", "subId", "Submission ID") if c in hist.columns),
    None,
)
if _sid:
    _per_y = hist.dropna(subset=["Date"]).assign(year=lambda d: d["Date"].dt.year)
    _yc = _per_y.groupby("year")[_sid].nunique().sort_index()
    if len(_yc) > 0:
        fig_y, ax_y = plt.subplots(figsize=(11, 4))
        _yc.plot(kind="bar", ax=ax_y, color="#3d7ea0")
        ax_y.set(title="Unique checklists per year", ylabel="checklists", xlabel="year")
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping per-year chart: no rows with a parsed Date.")

    if len(_per_y) > 0:
        _per_y = _per_y.assign(month=lambda d: d["Date"].dt.month)
        _mc = _per_y.groupby("month")[_sid].nunique()
        if len(_mc) > 0:
            fig_m, ax_m = plt.subplots(figsize=(11, 4))
            _mc.plot(kind="bar", ax=ax_m, color="#d83333")
            ax_m.set(
                title="Unique checklists per calendar month (all years)",
                ylabel="checklists",
                xlabel="month",
            )
            plt.tight_layout()
            plt.show()

_sci = "Scientific Name" if "Scientific Name" in hist.columns else "Scientific_name"
_top_sp = (
    hist.drop_duplicates(subset=[_sid, _sci] if _sid else [_sci])
    .groupby(_sci, dropna=True)
    .size()
    .sort_values(ascending=False)
    .head(15)
    .iloc[::-1]
)
fig_s, ax_s = plt.subplots(figsize=(11, 7))
ax_s.barh(_top_sp.index.astype(str), _top_sp.values, color="#457B9D")
ax_s.set(title="Top 15 species by checklist rows (deduped per checklist)")
plt.tight_layout()
plt.show()

if _sid and "Location ID" in hist.columns:
    _loc = (
        hist.drop_duplicates(subset=[_sid, "Location ID"])
        .groupby(["Location", "Location ID"])
        .size()
        .sort_values(ascending=False)
        .head(15)
        .iloc[::-1]
    )
    fig_l, ax_l = plt.subplots(figsize=(11, 8))
    ax_l.barh([str(i)[:70] for i in _loc.index], _loc.values, color="#6A4C93")
    ax_l.set(title="Top 15 locations by checklist count")
    plt.tight_layout()
    plt.show()



### Choropleth — your species by country (family search)


In [ ]:
from birds_history import personal_choropleth_html

_fe = df_family.set_index("Scientific_name")["Family_English_name"].to_dict()
_html = personal_choropleth_html(
    df_species,
    hist,
    _fe,
    api_key=os.environ.get("EBIRD_API_KEY"),
    cache_dir=EBIRD_CACHE,
)
display(HTML(_html))



### Regional overlaps — Southeast Asia vs Americas


In [ ]:
from birds_history import region_overlap_figures

_SEA = frozenset({"ID", "KH", "LA", "MY", "TH", "VN"})
_AME = frozenset({"BO", "CA", "CL", "GD", "MX", "PE", "US"})

_f1, _f2 = region_overlap_figures(hist, _SEA, region_title="Southeast Asia")
_f1.show()
_f2.show()

_g1, _g2 = region_overlap_figures(hist, _AME, region_title="Americas")
_g1.show()
_g2.show()



### PCA — countries in species space


In [ ]:
from birds_history import country_pca

_pc1, _pc2 = country_pca(hist, df_species, min_species_per_country=5)
_pc1.show()
_pc2.show()

